# Reporte de descripción de los datos de ordeño

En este notebook calculamos todas las cifras de ordeño para el *Reporte de descripción de los datos*. Para facilitar su lectura, hemos decidido dividir este notebook en las siguientes secciones:

| Sección | Qué hace | Parte del reporte en la que se usa |
| --- | --- | --- |
| 1. Configuración y carga | Se definen los parámetros, la ruta de los datos y lee los 34 archivos CSV y los une | 2.1 y 6 |
| 2. Descripción de los datos | Se describe qué es cada fila, estructura, formato y columnas | 2 y 3 |
| 3. Exploración de los datos | Se hacen los resúmenes por vaca, por cuarto, global, los incidentes, el destino de la leche y la cobertura | 4 y Anexo A |
| 4. Verificación de los datos | Es donde revisamos la calidad de los datos, comparamos las cifras del reporte contra lo que calculamos y listamos los pendientes | 5 |
| 5. Exportar resultados | Guardamos los resultados en un Excel | Anexo A |

## ¿Cómo ejecutar este código?

- Si se va a ejecutar en Colab, entonces en la Sección 1 debes montar el Dirve y usar por defecto la carpeta que nos compartió el socio formador con los datos de la vaca para respetar la privacidad de dichos datos,.
- Si se va a ejecutar fuera de Colab, entocnes se debe definir la ruta antes de abrir el notebook con la variable `DATOS_VACAS_DIR`o escribir la ruta en `RUTA_DATOS` de la sección 1. Los resultados se guardarán en `salidas/`.

## Notas importantes del manejo de los datos

Recordemos que tenemos las siguientes reglas y limitaciones:
- Los datos no se deben subir al repositorio, de modo que la carpeta salidas/ tendrá que estar en `.gitignore`.
- Los datos tampoco pueden guardarse fuera del Drive que se nos compartió ni de debe usar la IA con ellos. Cualquier manejo de rutas se tendrá que hacer con la autorización del socio formador y siguiendo los acuerdos de confidencialidad previamente definidos.

## 1. Configuración y carga

Esta parte contiene todo aquello que cambiaría si es que llegara a haber algún cambio con nuestras carptas o algún otro asunto. Solo se deben tocar los parámetros, nada más.

In [ ]:
import os, re, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

In [ ]:
# Parámetros
NOMBRE_CARPETA = "Compartidos conmigo/Datos/DATOS vacas en establo marzo_junio_2025"

VACAS = [1204, 1221, 1233, 1497, 1554, 1572, 1613, 1624, 1634,
         2072, 2076, 2078, 2082, 6003, 6036, 6178, 6191, 6194,
         6211, 6232, 6238, 8709, 8712, 8729, 8732, 8755, 8758,
         8771, 8775, 8777, 8779, 8782, 8792, 8794]

CUARTOS = ["DI", "DD", "TI", "TD"]
VENTANA_DIAS = 140
MIN_ORDENOS = 30 # mínimo de ordeños para comparar las vacas entre sí
UMBRAL_COND = 6.0 # mS/cm. Es el valor alto de conductividad


In [ ]:
# Rutas
RUTA_DATOS = None # ejemplo: "/ruta/a/la/carpeta/con/los/csv"
RUTA_SALIDA = None # carpeta donde se guardan los resultados (donde por defecto sería ./salidas)

In [ ]:
def resolver_ruta_datos():
    if RUTA_DATOS:
        return Path(RUTA_DATOS).expanduser()
    if os.environ.get("DATOS_VACAS_DIR"):
        return Path(os.environ["DATOS_VACAS_DIR"]).expanduser()
    if EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive") / NOMBRE_CARPETA
    raise FileNotFoundError(
        "No se indicó dónde están los datos. Define la variable de entorno DATOS_VACAS_DIR "
        "o escribe la ruta en RUTA_DATOS (sección 1).")

In [ ]:
def resolver_ruta_salida():
    ruta = Path(RUTA_SALIDA or os.environ.get("SALIDAS_VACAS_DIR", "salidas")).expanduser()
    ruta.mkdir(parents=True, exist_ok=True)
    return ruta

In [ ]:
def dentro_de_repo_git(ruta):
    ruta = Path(ruta).resolve()
    return any((p / ".git").exists() for p in [ruta, *ruta.parents])

In [ ]:
carpeta = resolver_ruta_datos()
if not carpeta.is_dir():
    raise FileNotFoundError(f"No existe la carpeta de datos: {carpeta}")
if dentro_de_repo_git(carpeta):
    warnings.warn("La carpeta de datos está dentro de un repositorio git: los datos no deben subirse al repositorio.")

salida = resolver_ruta_salida()
if dentro_de_repo_git(salida):
    warnings.warn("La carpeta de salidas está dentro de un repositorio git: agrégala a .gitignore.")

print("Entorno:", "Colab" if EN_COLAB else "local")
print("Carpeta de datos:", carpeta)
print("Carpeta de salidas:", salida.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: No existe la carpeta de datos: /content/drive/MyDrive/Compartidos conmigo/Datos/DATOS vacas en establo marzo_junio_2025

### 1.1. Localizar los archivos

In [ ]:
archivos = {}
for p in carpeta.rglob("*.csv"):
    m = re.search(r"\d{4}", p.stem)
    if m and int(m.group()) in VACAS:
        archivos[int(m.group())] = p

print(f"Encontrados {len(archivos)} de {len(VACAS)} archivos")
faltan = sorted(set(VACAS) - set(archivos))
print("Faltan:", faltan)

### 1.2. Funciones y carga

- Como cada CSV tiene un encabezado de dos niveles (que serían el grupo y el cuarto), la función `leer_vaca` combina dichos niveles en un solo nobmre para que sea más fácil de leer (por ejemplo, Sangre (ppm) | DI).
- Como las fechas vienen como texto, se transforman a tipo fecha y hora.
- La duración también viene como textoe n formato mm:ss, por lo que lo convertimos a segundos.

In [ ]:
num = lambda s: pd.to_numeric(s, errors="coerce")

In [ ]:
def leer_vaca(ruta):
    raw = pd.read_csv(ruta, header=None, dtype=str, encoding="utf-8-sig")
    grupos = raw.iloc[0].ffill()
    subs = raw.iloc[1].fillna("").str.strip()

    nombres = []
    for g, s in zip(grupos, subs):
        g = str(g).strip()
        if g == "Main":
            nombres.append(s or g)
        else:
            nombres.append(f"{g} | {s}" if s else g)

    df = raw.iloc[2:].reset_index(drop=True)
    df.columns = nombres

    # Convertir a número lo que se pueda (el resto queda como texto)
    for c in df.columns:
        convertida = pd.to_numeric(df[c].str.replace(",", ".", regex=False), errors="coerce")
        if convertida.notna().sum() > 0.5 * df[c].notna().sum():
            df[c] = convertida
    return df

In [ ]:
def parsear_fecha(serie):
    s = (serie.astype(str)
         .str.replace(" ", " ", regex=False)
         .str.replace("\xa0", " ", regex=False)
         .str.replace(r"a\.\s*m\.", "AM", regex=True)
         .str.replace(r"p\.\s*m\.", "PM", regex=True)
         .str.strip())
    return pd.to_datetime(s, format="%d/%m/%Y %I:%M %p", errors="coerce")

In [ ]:
def mmss_a_seg(x):
    if pd.isna(x):
        return np.nan
    try:
        p = [float(i) for i in str(x).split(":")]
    except ValueError:
        return np.nan
    if len(p) == 2:
        return p[0] * 60 + p[1]
    if len(p) == 3:
        return p[0] * 3600 + p[1] * 60 + p[2]
    return np.nan

In [ ]:
dfs = {}
for vaca, ruta in sorted(archivos.items()):
    d = leer_vaca(ruta)
    d["fecha"] = parsear_fecha(d["Hora de inicio"])
    d["dur_s"] = d["Duración (mm:ss)"].map(mmss_a_seg)
    d["Producción (kg)"] = num(d["Producción (kg)"])
    d = d.sort_values("fecha").reset_index(drop=True)
    d.insert(0, "vaca", vaca)
    dfs[vaca] = d

todas = pd.concat(dfs.values(), ignore_index=True)

# Solo los ordeños entran en las estadísticas de producción, incidentes y destino
dfs_ord = {v: d[d["Acción"] == "Ordeño"].reset_index(drop=True) for v, d in dfs.items()}
ord_total = pd.concat(dfs_ord.values(), ignore_index=True)

print("Vacas cargadas:", len(dfs), "| filas totales:", len(todas), "| ordeños:", len(ord_total))

## 2. Descripción de los datos

En esta sección se describe qué contienen los archivos (aunque aún no se intrpretan), donde revisamos su origen y alcance, qué es cada fiila, cuartos de la ubre y su formato. Con esto obtenemos la información para las partes 2 y 3 del reporte.

### 2.1. Origen y alcance

In [ ]:
print("Archivos leídos:", len(dfs))
print("Vacas:", sorted(dfs))
print("Filas totales:", len(todas))
print("Rango de fechas:", todas["fecha"].min(), "a", todas["fecha"].max())
print("Días en la ventana:", (todas["fecha"].max().normalize() - todas["fecha"].min().normalize()).days + 1)

### 2.2. Significado de cada fila

La columna de Acción es la que indica el tipo de evento. Solo aquellos registros con unas acción de Ordeño son ordeños, mientras que acciones como Rechazada o Echada sin ordeñar no lo son.

In [ ]:
acc = todas["Acción"].value_counts(dropna=False)
display(acc.to_frame("filas").assign(porcentaje=lambda x: (x["filas"] / len(todas) * 100).round(1)))

# Las filas que no son ordeño deberían ser las mismas que no tienen destino de la leche ni AMD de ubre
no_ord = todas["Acción"] != "Ordeño"
sin_destino = todas["Misc | Destino Leche"].isna()
sin_amd = todas["Estado AMD | Ubre"].isna()
print("Filas que no son ordeño:", int(no_ord.sum()))
print("Filas sin Destino Leche:", int(sin_destino.sum()), "| sin AMD de ubre:", int(sin_amd.sum()))
print("Coinciden exactamente con las no ordeño:",
      bool((no_ord == sin_destino).all() and (no_ord == sin_amd).all()))

### 2.3. Cuartos de la ubre

Hay varias columnas que se repiten una vez por cuarto que son:
- DI: delantero izquierdo
- DD: delantero derecho
- TI: trasero izquierdo
- TD: trasero derecho

In [ ]:
grupos_cuartos = {}
for c in todas.columns:
    base, _, sufijo = c.rpartition(" | ")
    if sufijo in CUARTOS:
        grupos_cuartos.setdefault(base, {})[sufijo] = c
display(pd.DataFrame(grupos_cuartos).T[CUARTOS])

### 2.4. Formato del archivo y columnas

Verificamos que los 34 archivos CSV tengan el mismo esquema, que las fechas sí sean legibles y también revisamos qué porcentaje de cada columnas está vacío (solo en las filas de ordeño).

In [ ]:
esquemas = {tuple(d.columns) for d in dfs.values()}
print("Esquemas de columnas distintos:", len(esquemas))
print("Número de columnas (sin las que agrega el notebook):", len([c for c in todas.columns if c not in ("vaca", "fecha", "dur_s")]))
print("Fechas no legibles:", int(todas["fecha"].isna().sum()))

nulos = pd.DataFrame({v: d.drop(columns="vaca", errors="ignore").isna().mean() * 100
                      for v, d in dfs_ord.items() if len(d) > 0}).T.round(1)

vacias_en_todas = nulos.columns[(nulos == 100).all()].tolist()
print("Columnas vacías en TODAS las vacas:", vacias_en_todas)

nulos_col = (ord_total.drop(columns="vaca", errors="ignore").isna().mean() * 100).round(1)
display(nulos_col.sort_values(ascending=False).to_frame("% vacío (ordeños)"))

In [ ]:
# Valores que toman las columnas categóricas principales
for col in ["Misc | EO/PO", "Misc | Destino Leche", "Misc | Razón de la desviación", "Estado AMD | Ubre"]:
    print(f"\n{col}:")
    print(todas[col].value_counts(dropna=False).to_string())

## 3. Exploración de los datos

Aquí hacemos los resúmenes por vaca, por cuarto y globales. Aquí obtenemos la información para la parte 4 del reporte y el Anexo A.

**Aclaración:** Solo se usan las filas de ordeño.

### 3.1. Resumen por vaca

In [ ]:
def resumir(d):
    f = d["fecha"]
    n = len(d)
    dias = f.dt.normalize().nunique()
    span = (f.max().normalize() - f.min().normalize()).days + 1
    inter_h = f.diff().dt.total_seconds() / 3600
    prod = d["Producción (kg)"]
    r = {
        "vaca": d["vaca"].iloc[0],
        "ordeños": n,
        "inicio": f.min().date(),
        "fin": f.max().date(),
        "días_calendario": span,
        "días_con_datos": dias,
        "cobertura_%": (dias / span * 100) if span > 0 else np.nan,
        "ordeños_por_día": (n / dias) if dias > 0 else np.nan,
        "intervalo_mediano_h": inter_h.median(),
        "huecos_>48h": int((inter_h > 48).sum()),
        "hueco_max_días": (inter_h.max() / 24) if len(inter_h) > 1 else np.nan,
        "cobertura_ventana_%": (dias / VENTANA_DIAS * 100),
        "datos_suficientes": n >= MIN_ORDENOS,
        "dur_media_min": d["dur_s"].mean() / 60,
        "prod_media_kg": prod.mean(),
        "prod_std_kg": prod.std(),
        "prod_min_kg": prod.min(),
        "prod_max_kg": prod.max(),
        "prod_diaria_kg": (prod.sum() / dias) if dias > 0 else np.nan,
    }
    for col, nombre in [("Estado | Patada", "patada"),
                        ("Estado | Incompleto", "incompleto"),
                        ("Estado | Pezones no encontrados", "pezones_no_enc")]:
        r[f"{nombre}_%"] = d[col].notna().mean() * 100
    dest = d["Misc | Destino Leche"]
    r["leche_a_tanque_%"] = ((dest == "Tanque").sum() / dest.notna().sum() * 100
                             if dest.notna().any() else np.nan)
    sin_vaca = d.drop(columns="vaca", errors="ignore")
    r["columnas_vacías"] = int(sin_vaca.isna().all().sum())
    r["nulos_%"] = sin_vaca.isna().mean().mean() * 100
    return r


acciones = pd.crosstab(todas["vaca"], todas["Acción"])
resumen = pd.DataFrame([resumir(d) for d in dfs_ord.values() if len(d) > 0]).round(2)
resumen = resumen.merge(acciones, left_on="vaca", right_index=True, how="left")
resumen

### 3.2. Descripción global

- En la columna Global se pondera cada vaca por su número de ordeños (esto se hace sobre todos los ordeños juntos)
- El resto de las columnas comparan las vaas entre sí y se usan solo las vacas que tengan al menos un cierto número mínimo de ordeños.

In [ ]:
comparables = resumen[resumen["datos_suficientes"]]
dias_total = resumen["días_con_datos"].sum()
n_ord = len(ord_total)

def fila(nombre, global_, serie):
    return {"Variable": nombre, "Global": global_, "Media entre vacas": serie.mean(),
            "Mediana entre vacas": serie.median(), "Mínimo": serie.min(), "Máximo": serie.max()}

tabla_global = pd.DataFrame([
    fila("Ordeños por vaca (todas las vacas)", n_ord, resumen["ordeños"]),
    fila("Producción por ordeño (kg)", ord_total["Producción (kg)"].mean(), comparables["prod_media_kg"]),
    fila("Producción diaria (kg)", ord_total["Producción (kg)"].sum() / dias_total, comparables["prod_diaria_kg"]),
    fila("Ordeños por día", n_ord / dias_total, comparables["ordeños_por_día"]),
    fila("Duración media (min)", ord_total["dur_s"].mean() / 60, comparables["dur_media_min"]),
    fila("Ordeños con patada (%)", ord_total["Estado | Patada"].notna().mean() * 100, comparables["patada_%"]),
    fila("Ordeños incompletos (%)", ord_total["Estado | Incompleto"].notna().mean() * 100, comparables["incompleto_%"]),
    fila("Ordeños con pezones no encontrados (%)",
         ord_total["Estado | Pezones no encontrados"].notna().mean() * 100, comparables["pezones_no_enc_%"]),
]).set_index("Variable").round(2)
tabla_global

### 3.3. Producción, frecuencia y duración para las vacas en los extremos

Se nombra a las vacas con los valores más altos y más bajos.

In [ ]:
def extremo(col, nombre, solo_comparables=True, decimales=2):
    base = comparables if solo_comparables else resumen
    base = base.dropna(subset=[col])
    if base.empty:
        return
    alto, bajo = base.loc[base[col].idxmax()], base.loc[base[col].idxmin()]
    print(f"{nombre}: máximo {alto[col]:.{decimales}f} (vaca {int(alto['vaca'])}) | "
          f"mínimo {bajo[col]:.{decimales}f} (vaca {int(bajo['vaca'])})")

extremo("prod_media_kg", "Producción media por ordeño (kg)")
extremo("dur_media_min", "Duración media (min)")
extremo("ordeños_por_día", "Ordeños por día")
extremo("intervalo_mediano_h", "Intervalo mediano (h)", decimales=1)
extremo("prod_diaria_kg", "Producción diaria (kg)", decimales=1)

mayor = ord_total.loc[ord_total["Producción (kg)"].idxmax()]
print(f"Ordeño más grande: {mayor['Producción (kg)']:.1f} kg (vaca {int(mayor['vaca'])})")

casi_cero = ord_total[ord_total["Producción (kg)"] <= 0.1]
print("Vacas con ordeños de 0 a 0.1 kg:",
      casi_cero.groupby("vaca")["Producción (kg)"].agg(["count", "min"]).round(2).to_dict("index"))

### 3.4. Comportamiento por cuarto

- `cuartos` tiene un resumen por vaca y por cuarto, de modo que cada vaca pesa igual.
- cuarto_pond calcula sobre todos los ordeños juntos (donde cada vaca opesa según sus ordeños). Las vaas que tengan un cuarto sin lectura quedan fuera de ese cuarto automáticamente para no meter valores vacíos.

In [ ]:
def resumir_cuartos(d):
    filas = []
    for q in CUARTOS:
        cond = num(d[f"Conductividad (mS / cm) | {q}"])
        sang = num(d[f"Sangre (ppm) | {q}"])
        filas.append({
            "vaca": d["vaca"].iloc[0],
            "cuarto": q,
            "prod_media_kg": num(d[f"Producciones (kg) | {q}"]).mean(),
            "flujo_medio": num(d[f"Media de los flujos (kg/min) | {q}"]).mean(),
            "flujo_max": num(d[f"Flujos máximos (kg/min) | {q}"]).mean(),
            "cond_media": cond.mean(),
            f"cond_>{UMBRAL_COND}_%": (cond > UMBRAL_COND).mean() * 100,
            "sangre_>0_n": int((sang > 0).sum()),
        })
    return filas

In [ ]:
cuartos = pd.DataFrame([f for d in dfs_ord.values() if len(d) > 0
                        for f in resumir_cuartos(d)]).round(2)

filas = []
for q in CUARTOS:
    cond = num(ord_total[f"Conductividad (mS / cm) | {q}"])
    sang = num(ord_total[f"Sangre (ppm) | {q}"])
    filas.append({
        "cuarto": q,
        "prod_media_kg": num(ord_total[f"Producciones (kg) | {q}"]).mean(),
        "flujo_mediano": num(ord_total[f"Media de los flujos (kg/min) | {q}"]).median(),
        "flujo_max_medio": num(ord_total[f"Flujos máximos (kg/min) | {q}"]).mean(),
        "cond_media": cond.mean(),
        f"ordeños_cond_>{UMBRAL_COND}_%": (cond > UMBRAL_COND).sum() / cond.notna().sum() * 100,
        "ordeños_con_sangre_>0": int((sang > 0).sum()),
    })
cuartos_pond = pd.DataFrame(filas).set_index("cuarto").round(2)
display(cuartos_pond)

# Vacas con asimetrías fuertes entre cuartos (cociente máximo/mínimo de producción media)
asim = (cuartos.pivot(index="vaca", columns="cuarto", values="prod_media_kg")
        .assign(razon_max_min=lambda x: x.max(axis=1) / x.min(axis=1)))
display(asim.sort_values("razon_max_min", ascending=False).head(5).round(2))

### 3.5. Incidentes de ordeño

Las columnas de Patada, Incompleto, Pezones no encontrados y AMD Pezón no son 0 o 1, sino que contienen el cuarto que fue afectado o varios separados por una coma. Si están vacías es porque no ocurrió el evento.

In [ ]:
columnas_incidente = ["Estado | Patada", "Estado | Incompleto", "Estado | Pezones no encontrados", "Estado AMD | Pezón"]

tabla_inc = {}
combinaciones = {}
for c in columnas_incidente:
    s = (todas[c].dropna().astype(str).str.split(",")
         .map(lambda l: tuple(sorted(x.strip() for x in l))))
    tabla_inc[c.split(" | ")[-1]] = {
        **{f"{q} (solo ese cuarto)": int((s == (q,)).sum()) for q in CUARTOS},
        "Varios cuartos": int((s.map(len) > 1).sum()),
        "Total de registros con el evento": int(len(s)),
    }
    combinaciones[c] = s[s.map(len) > 1].value_counts().head(5)

display(pd.DataFrame(tabla_inc))
for c, v in combinaciones.items():
    print(f"\nCombinaciones más frecuentes de {c}:")
    print(v.to_string())

### 3.6. Estado AMD

Estado AMD | Ubre tom los valores 0 y 1 (aunque aún nos falta confirmar qué significa 1, de modo que solo vamos a reportar su frecuencia).

In [ ]:
ubre = num(ord_total["Estado AMD | Ubre"])
print(f"AMD ubre = 1 en {int((ubre == 1).sum())} de {len(ord_total)} ordeños ({(ubre == 1).mean() * 100:.1f}%)")
print("AMD pezón con cuarto indicado:", int(todas["Estado AMD | Pezón"].notna().sum()), "registros")

### 3.7. Destino de la leche y robots (VMS)

In [ ]:
destino = ord_total["Misc | Destino Leche"].value_counts()
destino_tab = destino.to_frame("ordeños").assign(porcentaje=lambda x: (x["ordeños"] / x["ordeños"].sum() * 100).round(1))
display(destino_tab)

desviados = int((ord_total["Misc | Destino Leche"] != "Tanque").sum())
print("Ordeños desviados (no fueron al tanque):", desviados)

por_vaca_tanque = resumen[["vaca", "ordeños", "leche_a_tanque_%"]]
print("Vacas con menos de 95% de su leche al tanque:", int((por_vaca_tanque["leche_a_tanque_%"] < 95).sum()))
display(por_vaca_tanque.sort_values("leche_a_tanque_%").head(5))

print("\nRegistros por unidad VMS (sobre todas las filas):")
vms = todas["Misc | EO/PO"].value_counts()
display(vms.to_frame("filas").assign(porcentaje=lambda x: (x["filas"] / len(todas) * 100).round(1)))

### 3.8. Cobertura temporal

Se abarcan varios tipo de cobertura:
- **Cobertura propia:** son los días con datos entre los días del periodo de esa vaca.
- **Cobertura de ventana:** son los días con datos entre VENTANA_DIAS (140).
- Un **hueco** es un periodo de más de 48 horas entre ordeños.

In [ ]:
inicio_global = todas["fecha"].min().date()

completa = resumen[(resumen["cobertura_%"] >= 97) & (resumen["cobertura_ventana_%"] >= 97)]
tardia = resumen[resumen["inicio"] > inicio_global]
ausencias = resumen[resumen["cobertura_%"] < 60].sort_values("cobertura_%")

print("Cobertura completa (>= 97%):", len(completa), "vacas:", completa["vaca"].tolist())
print("Entrada tardía (empiezan después del", inicio_global, "):", len(tardia), "vacas")
print("Ausencias largas (cobertura propia < 60%):", len(ausencias))
display(ausencias[["vaca", "cobertura_%", "huecos_>48h", "hueco_max_días"]])

### 3.9. Gráfica

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 5))
resumen.set_index("vaca")["ordeños"].plot.bar(ax=ax[0], title="Ordeños por vaca")
ord_total.boxplot(column="Producción (kg)", by="vaca", ax=ax[1], rot=90)
ax[1].set_title("Producción por ordeño (kg)")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 4. Verificación de los datos

Aquói revisamos la calidad de los datos y comparamos las cifras del reporte con los de notebook. Esta parte aporta información para la sección 5 del reporte. Si hay cifras que no corresponden en el reporte, se corrigen o se hace una auditoría del dato.

### 4.1. Cifras del reporte contra lo calculado

Al ser esta la segunda versión del código, creada luego de las correcciones del reporte, vamos a tomar de referencia los datos del reporte para vrificar que cada dato que hayamos colocado sea el correcto.

In [ ]:
cond_cuartos_sin_lectura = cuartos[cuartos["prod_media_kg"].isna()]
dup_mask = todas.duplicated(subset=["vaca", "Hora de inicio"], keep="first")

verificaciones = [
    ("Archivos leídos", 34, len(dfs)),
    ("Filas totales", 7249, len(todas)),
    ("Filas de ordeño", 7016, int((todas["Acción"] == "Ordeño").sum())),
    ("Filas Rechazada", 172, int((todas["Acción"] == "Rechazada").sum())),
    ("Filas Echada sin ordeñar", 61, int((todas["Acción"] == "Echada sin ordeñar").sum())),
    ("Fechas no legibles", 0, int(todas["fecha"].isna().sum())),
    ("Esquemas de columnas distintos", 1, len({tuple(d.columns) for d in dfs.values()})),
    ("Registros con vaca y hora de inicio repetidos", 25, int(dup_mask.sum())),
    ("Ordeños al tanque", 6772, int((ord_total["Misc | Destino Leche"] == "Tanque").sum())),
    ("Ordeños a Drenaje", 152, int((ord_total["Misc | Destino Leche"] == "Drenaje").sum())),
    ("Ordeños a Divert 3", 92, int((ord_total["Misc | Destino Leche"] == "Divert 3").sum())),
    ("Ordeños desviados", 244, desviados),
    ("Vacas con al menos 30 ordeños", 31, int(resumen["datos_suficientes"].sum())),
    ("Vacas con cobertura completa", 14, len(completa)),
    ("Vacas con entrada tardía", 12, len(tardia)),
    ("Vacas con ausencias largas (< 60%)", 6, len(ausencias)),
    ("Cuartos sin ninguna lectura", 2, len(cond_cuartos_sin_lectura)),
]
ver = pd.DataFrame(verificaciones, columns=["Cifra", "En el reporte", "Calculado"])
ver["Coincide"] = np.where(ver["En el reporte"] == ver["Calculado"], "✓", "✗")
display(ver)
if len(dfs) < len(VACAS):
    print("\nAVISO: no se cargaron todos los archivos; es normal que varias cifras no coincidan.")

### 4.2. Posibles duplicados

Hay registros que comparten vaca y hora de inicio, aspi que habría que revisar si se tratan de copias exactas o de registros distintos. De momento no las vamos a eliminar, de modo que los conteos de estas filas podrían estar infladas.

In [ ]:
dup = todas[todas.duplicated(subset=["vaca", "Hora de inicio"], keep=False)].sort_values(["vaca", "fecha"])
print(len(dup), "filas involucradas | copias exactas (todas las columnas):",
      int(todas.drop(columns=["fecha", "dur_s"]).duplicated(keep=False).sum()))
print("Duplicados por vaca:", dup["vaca"].value_counts().to_dict())
display(dup[["vaca", "Hora de inicio", "Acción", "Producción (kg)", "Estado | Número de ordeño"]].head(20))

### 4.3. Producciones en cero y valores extremos

Auditamos los ordeños con 0 kg y los valores imposibles (es decir, valores muy exagerados de producción) para poder revisar dichos casos.

In [ ]:
print(ord_total["Producción (kg)"].describe().round(2).to_string())
print("\nOrdeños con producción <= 0.1 kg:")
display(ord_total[ord_total["Producción (kg)"] <= 0.1][["vaca", "Hora de inicio", "Duración (mm:ss)", "Producción (kg)"]])
print("Los 10 ordeños más grandes:")
display(ord_total.nlargest(10, "Producción (kg)")[["vaca", "Hora de inicio", "Duración (mm:ss)", "Producción (kg)"]])

### 4.4. Cuartos sin lectura y vacas con pocos datos

In [ ]:
print("Cuartos sin ninguna lectura (según el socio, cuartos que no se ordeñan):")
display(cond_cuartos_sin_lectura[["vaca", "cuarto"]])

print("Vacas con menos de", MIN_ORDENOS, "ordeños (excluidas de las comparaciones):")
display(resumen.loc[~resumen["datos_suficientes"], ["vaca", "ordeños", "inicio", "fin"]])

criticas = [v for v in (6211, 6232, 1497) if v in nulos.index]
if criticas:
    x = nulos.loc[criticas].T
    print("Columnas con más de 50% de vacíos en vacas críticas:")
    display(x[(x > 50).any(axis=1)])

### 4.5. Incidentes conta acciones y columnas vacías

Aquellas columnas de incidente que están vacías significa que el evento no ocurrió, por lo que vamos a revisar que dichos valores vacíos se concentren como deberían.

In [ ]:
for c in ["Estado | Patada", "Estado | Incompleto", "Estado | Pezones no encontrados"]:
    print(f"\n{c}: % de filas con el evento, por Acción")
    print(pd.crosstab(todas["Acción"], todas[c].notna(), normalize="index").mul(100).round(1).to_string())

print("\nDestino de la leche en vacas con pocos datos o muy desviadas:")
destinos = pd.crosstab(todas["vaca"], todas["Misc | Destino Leche"])
display(destinos.reindex([v for v in (6003, 8771, 8775, 1497) if v in destinos.index]))

### 4.6. Conductividad y umbral

Consideramos la conductividad como alta si se supera el umbral de 6.0 mS/cm.

Si se llega a cambiar el umbral, modificar `UMBRAL_COND` en la sección 1 y volver a ejecutar esta sección.

In [ ]:
print("Umbral de conductividad usado:", UMBRAL_COND, "mS/cm")
display(cuartos_pond[[c for c in cuartos_pond.columns if c.startswith("ordeños_cond")]])

## 5. Exportar resultados

Guarda las tablas en un excelk dento de la carpeta de salidas (`salidas/` por defecto).

IMPORTANTE: No se exportan los datos crudos.

La última celda es para poder descargar dicho excel.

In [ ]:
archivo_xlsx = salida / "resumen_vacas.xlsx"
global_vacas = (resumen.drop(columns=["vaca", "inicio", "fin"], errors="ignore")
                .describe().T[["mean", "std", "min", "50%", "max"]].round(2))
global_cuartos = cuartos.drop(columns="vaca", errors="ignore").groupby("cuarto").mean().round(2)

with pd.ExcelWriter(archivo_xlsx) as w:
    resumen.to_excel(w, sheet_name="por_vaca", index=False)
    cuartos.to_excel(w, sheet_name="por_cuarto", index=False)
    cuartos_pond.to_excel(w, sheet_name="cuartos_ponderado")
    tabla_global.to_excel(w, sheet_name="tabla_global")
    global_vacas.to_excel(w, sheet_name="global_vacas")
    global_cuartos.to_excel(w, sheet_name="global_cuartos")
    nulos.to_excel(w, sheet_name="nulos_%")
    ver.to_excel(w, sheet_name="verificacion", index=False)

print("Guardado en:", archivo_xlsx.resolve())

In [ ]:
if EN_COLAB:
    from google.colab import files
    files.download(str(archivo_xlsx))
else:
    print("Fuera de Colab: el archivo ya está en", archivo_xlsx.resolve())